In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
target_size = (256, 256)


In [ ]:
from datautils import datautils
datautil = datautils()


In [ ]:

X_train_rgb, y_train, X_val_rgb, y_val = datautil.load_data(target_size)

In [ ]:
from dicomutils import dicomutils
utils = dicomutils()


In [ ]:
# ResNet eğitimi
print("ResNet50 eğitiliyor...")
resnet_history = utils.fit_resnet_model(X_train_rgb, y_train, X_val_rgb, y_val)




In [ ]:
# EfficientNet eğitimi
print("\nEfficientNetB4 eğitiliyor...")
effnet_history =utils.fit_efficientnet_model(X_train_rgb, y_train, X_val_rgb, y_val)

In [ ]:
from figureutils import  figureutils
fig= figureutils()


In [ ]:

fig.plot_history(resnet_history, 'ResNet50')


In [ ]:
fig.plot_history(effnet_history, 'EfficientNetB4')

In [ ]:
test_images = utils.load_dicom_data("YarısmaVeriSeti_1.Oturum/DICOM")


In [ ]:
# Test verisini hazırlama
test_images_rgb = np.repeat(test_images[..., np.newaxis], 3, axis=-1)


In [ ]:
resnet_model = tf.keras.models.load_model("best_model_ResNet50")

In [ ]:
effnet_model = tf.keras.models.load_model("best_model_effnet_model")

In [ ]:
class_names = ['Kanama', 'İskemi', 'Normal']

In [ ]:

# ResNet tahminleri
resnet_preds = resnet_model.predict(test_images_rgb)
resnet_classes = resnet_preds.argmax(axis=1)

# EfficientNet tahminleri
effnet_preds = effnet_model.predict(test_images_rgb)
effnet_classes = effnet_preds.argmax(axis=1)

# Sonuçları kaydetme
for i, (img, resnet_class, effnet_class) in enumerate(zip(test_images, resnet_classes, effnet_classes)):
    with open(f"YarısmaVeriSeti_1.Oturum/MASKS/{i}_advanced_preds.txt", "w") as f:
        f.write(f"ResNet50 Tahmini: {class_names[resnet_class]}\n")
        f.write(f"EfficientNetB4 Tahmini: {class_names[effnet_class]}\n")
        f.write("\nResNet50 Olasılıklar:\n")
        for name, prob in zip(class_names, resnet_preds[i]):
            f.write(f"{name}: {prob:.4f}\n")
        f.write("\nEfficientNetB4 Olasılıklar:\n")
        for name, prob in zip(class_names, effnet_preds[i]):
            f.write(f"{name}: {prob:.4f}\n")

In [ ]:
#Ince Ayar (Fine-Tuning): Birkaç epoch sonra bazı katmanları açarak ince ayar yapabilirsiniz:
# for layer in base_model.layers[-10:]:
#     layer.trainable = True
# model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # Düşük learning rate
#               loss='sparse_categorical_crossentropy',
#               metrics=['accuracy'])

In [ ]:
#Veri Artırma: Daha güçlü bir pipeline oluşturun:
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# datagen = ImageDataGenerator(
#     rotation_range=20,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     shear_range=0.2,
#     zoom_range=0.2,
#     horizontal_flip=True,
#     fill_mode='nearest'
# )

En İyi Modelin Seçimi ve Optimizasyon:

In [ ]:
# Model performanslarını karşılaştır
resnet_test_loss, resnet_test_acc = resnet_model.evaluate(X_val_rgb, y_val)
effnet_test_loss, effnet_test_acc = effnet_model.evaluate(X_val_rgb, y_val)

best_model = resnet_model if resnet_test_acc > effnet_test_acc else effnet_model
print(f"Seçilen model: {'ResNet50' if resnet_test_acc > effnet_test_acc else 'EfficientNetB4'}")

# Modeli kaydet
best_model.save('best_stroke_classifier.h5')

Hiperparametre Optimizasyonu (Opsiyonel):

In [ ]:
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import GridSearchCV



# Parametre grid'i
param_grid = {
    'learning_rate': [0.001, 0.0001],
    'dense_units': [512, 1024, 2048]
}

eficientnet_model= utils.build_efficientnet_model()
# GridSearchCV

model = KerasClassifier(build_fn=eficientnet_model, epochs=10, batch_size=16)
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)
grid_result = grid.fit(X_train_rgb, y_train)

# En iyi parametreler
print(f"En iyi doğruluk: {grid_result.best_score_} ile {grid_result.best_params_}")

Model Interpretability (Grad-CAM ile Görselleştirme):

In [ ]:


# Örnek bir görüntü için Grad-CAM
sample_img = X_val_rgb[0][np.newaxis, ...]

fig.make_gradcam_heatmap(sample_img, best_model, 'conv5_block3_out')


Üretime Hazırlık:

In [ ]:
# TF-Lite modeline dönüştürme
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

